
# Derivación tensorial **realmente simbólica** para teorías $f(R)$

Este notebook no usa `show_eq(...)` para “contar” la deducción. Cada línea importante es un **objeto simbólico real** almacenado en `S[...]`.

La regla de trabajo es:

$
L_{\rm input}
\longrightarrow
\frac{\partial L}{\partial R}
\longrightarrow
P^{abcd}
\longrightarrow
P^{ab},\ \mathcal R_{ab}
\longrightarrow
\delta(\sqrt{-g}L)
\longrightarrow
\text{Palatini}
\longrightarrow
\text{dos IBP}
\longrightarrow
E_{ab}.
$

En particular:

- las simetrías de $P^{abcd}$ se obtienen aplicando proyectores algebraicos;
- los cuatro términos de $\mathcal L_\xi R_{abcd}$ se canonizan uno por uno;
- el reemplazo de $\delta\Gamma$ se expande en tres términos y el CAS prueba cuáles se anulan o coinciden;
- cada integración por partes se verifica expandiendo la divergencia con la regla del producto;
- $-2\nabla^m\nabla^nP_{amnb}$ se calcula desde el $P$ generado por el input.

Todo queda disponible para reutilizarse, por ejemplo `S["P_abcd"]`, `S["lie_curv_term_3"]`, `S["ibp1_divergence"]`, `S["E_ab_raw"]`, etc.


In [17]:

import sympy as sp
from IPython.display import display, Markdown
from sympy.tensor.tensor import (
    TensorIndexType, TensorHead, TensorSymmetry,
    tensor_indices, canon_bp, contract_metric, TensExpr
)

sp.init_printing()

# ================================================================
# 1. Geometría abstracta
# ================================================================
M = TensorIndexType(
    "M",
    dummy_name="0",
    metric_symmetry=1,
    metric_name=r"\mathrm{g}",
)
g = M.metric

sym0 = TensorSymmetry.no_symmetry
symS = TensorSymmetry.fully_symmetric
symDP = TensorSymmetry.direct_product

Riem = TensorHead(r"\mathrm{R}", [M]*4, TensorSymmetry.riemann())
dRiem = TensorHead(r"\delta\mathrm{R}", [M]*4, TensorSymmetry.riemann())

xi = TensorHead(r"\xi", [M], sym0(1))
Dxi = TensorHead(r"\nabla\xi", [M]*2, sym0(2))

# ∇_m R_{abcd}: no se impone una simetría adicional en el índice derivativo.
DRiem = TensorHead(r"\nabla\mathrm{R}", [M]*5, sym0(5))

# Variación métrica y sus derivadas
h = TensorHead(r"\delta\mathrm{g}", [M]*2, symS(2))          # h_ab = δg_ab
H = TensorHead(r"\mathrm{H}", [M]*2, symS(2))          # H^ab = δg^ab
Dh = TensorHead(r"\nabla\delta\mathrm{g}", [M]*3, symDP(1, 2))    # ∇_a h_bc
DDh = TensorHead(r"\nabla\nabla\delta\mathrm{g}", [M]*4, symDP(1, 1, 2)) # ∇_a∇_b h_cd

# Variación de la conexión y su derivada
DGamma = TensorHead(r"\nabla\delta\Gamma", [M]*4, symDP(1, 1, 2))
# DGamma^e{}_{cdb} representa ∇_c δΓ^e{}_{db}

# Gradiente y Hessiano del escalar R
DR = TensorHead(r"\nabla R", [M], sym0(1))          # ∇_a R
DDR = TensorHead(r"\nabla\nabla R", [M]*2, symS(2))      # ∇_a∇_b R

sqrtg = sp.Symbol("sqrt_minus_g", positive=True)

# ================================================================
# 2. Símbolos escalares permitidos en el input
# ================================================================
R = sp.Symbol("R", real=True)
alpha, beta, gamma, Lambda, mu = sp.symbols(
    "alpha beta gamma Lambda mu", real=True
)

# ================================================================
# 3. Utilidades de álgebra tensorial
# ================================================================
def tsimplify(expr, max_iter=8):
    """Canoniza índices mudos, contrae métricas y simplifica coeficientes."""
    if expr == 0:
        return sp.S.Zero

    if isinstance(expr, TensExpr):
        cur = sp.expand(expr)
        prev = None
        for _ in range(max_iter):
            cur = canon_bp(cur)
            cur = contract_metric(cur, g)
            cur = canon_bp(cur)
            if cur == prev:
                break
            prev = cur
        return cur

    return sp.factor(sp.simplify(expr))


def swap(expr, i, j):
    """Intercambio simultáneo de dos índices libres."""
    return expr.xreplace({i: j, j: i})


def antisym(expr, i, j):
    return sp.Rational(1, 2) * (expr - swap(expr, i, j))


def pair_exchange(expr, a, b, c, d):
    repl = {a: c, b: d, c: a, d: b}
    return expr.xreplace(repl)


def pair_sym(expr, a, b, c, d):
    return sp.Rational(1, 2) * (expr + pair_exchange(expr, a, b, c, d))


def curvature_projector(a, b, c, d, return_steps=False):
    """
    Parte de Q^{abcd}=g^{ac}g^{bd} y proyecta:
      1) antisimetría en (a,b),
      2) antisimetría en (c,d),
      3) simetría por intercambio de pares.
    """
    q0 = g(a, c) * g(b, d)
    q1 = tsimplify(antisym(q0, a, b))
    q2 = tsimplify(antisym(q1, c, d))
    q3 = tsimplify(pair_sym(q2, a, b, c, d))
    if return_steps:
        return q0, q1, q2, q3
    return q3


def dginv_dgcov(x, y, p, q):
    """
    ∂g^{xy}/∂g_{pq}, respetando que g_{pq}=g_{qp}.
    """
    return -sp.Rational(1, 2) * (
        g(x, p)*g(y, q) + g(x, q)*g(y, p)
    )


def h_from_H(a, b):
    """δg_ab expresada en función de δg^mn."""
    m, n = tensor_indices("hH_m hH_n", M)
    return -g(-a, -m) * g(-b, -n) * H(m, n)


def scalar_covd(expr, a):
    """∇_a expr(R) por regla de la cadena."""
    return sp.diff(expr, R) * DR(-a)


def scalar_hessian(expr, a, b):
    """∇_a∇_b expr(R) por regla de la cadena."""
    return (
        sp.diff(expr, R) * DDR(-a, -b)
        + sp.diff(expr, R, 2) * DR(-a) * DR(-b)
    )


class StepStore(dict):
    def put(self, key, expr, simplify=True):
        self[key] = tsimplify(expr) if simplify else expr
        return self[key]

    def show(self, key, title=None):
        if title is None:
            title = key
        display(Markdown(f"#### {title}\nObjeto reutilizable: `S[{key!r}]`"))
        display(self[key])
        return self[key]

    def check_zero(self, key, expr, title=None):
        result = self.put(key, expr)
        if title is None:
            title = key
        display(Markdown(f"#### Verificación: {title}\nObjeto: `S[{key!r}]`"))
        display(result)
        if result != 0:
            raise AssertionError(f"La verificación {key} no dio cero.")
        return result


S = StepStore()


## Input del usuario

Cambia **solo** `L_input`. El resto del notebook vuelve a calcular toda la cadena.

También funciona con una función no especificada:

```python
f = sp.Function("f")
L_input = f(R)
```


In [18]:

# ================================================================
# EDITAR SOLO ESTA LÍNEA
# ================================================================
#L_input = R + alpha*R**2 - 2*Lambda
L_input = R

L_input = sp.sympify(L_input)

if L_input.has(sp.Derivative):
    raise ValueError("El input debe depender de R, sin derivadas explícitas de R.")

f1 = sp.simplify(sp.diff(L_input, R))
f2 = sp.simplify(sp.diff(L_input, R, 2))
f3 = sp.simplify(sp.diff(L_input, R, 3))

S.put("L_input", L_input)
S.put("f_R", f1)
S.put("f_RR", f2)
S.put("f_RRR", f3)

display(Markdown("### Datos escalares calculados desde el input"))
for key in ("L_input", "f_R", "f_RR", "f_RRR"):
    S.show(key)


### Datos escalares calculados desde el input

#### L_input
Objeto reutilizable: `S['L_input']`

#### f_R
Objeto reutilizable: `S['f_R']`

#### f_RR
Objeto reutilizable: `S['f_RR']`

#### f_RRR
Objeto reutilizable: `S['f_RRR']`


# Etapa 1. Construir $P^{abcd}$ desde $L_{\rm input}$

No se escribe la fórmula final. Se parte del coeficiente ingenuo

$Q_0^{abcd}=g^{ac}g^{bd}$

y se proyecta algebraicamente al espacio con las simetrías de Riemann.


In [19]:

a, b, c, d = tensor_indices("a b c d", M)

q0, q1, q2, q3 = curvature_projector(a, b, c, d, return_steps=True)

S.put("Q_naive", q0)
S.put("Q_antisym_ab", q1)
S.put("Q_antisym_ab_cd", q2)
S.put("dR_dRiemann", q3)

# Regla de la cadena calculada desde el input:
# P^{abcd} = (dL/dR) * (dR/dR_abcd)
S.put("P_abcd", f1 * q3)

for key, title in [
    ("Q_naive", "Coeficiente antes de imponer simetrías"),
    ("Q_antisym_ab", "Después de antisimetrizar el primer par"),
    ("Q_antisym_ab_cd", "Después de antisimetrizar ambos pares"),
    ("dR_dRiemann", "Después de simetrizar el intercambio de pares"),
    ("P_abcd", "P^{abcd} calculado desde L_input"),
]:
    S.show(key, title)

# Función usada por las siguientes etapas; siempre reconstruye el P
# a partir del proyector calculado y del f_R del input.
P_TEMPLATE_INDICES = (a, b, c, d)
P_PROJECTOR_TEMPLATE = q3
P_TEMPLATE = f1*q3

def _reindex4(expr, old, new):
    return expr.xreplace(dict(zip(old, new)))

def P_up(a, b, c, d):
    # Reutiliza el proyector que ya fue CALCULADO arriba.
    return _reindex4(P_TEMPLATE, P_TEMPLATE_INDICES, (a,b,c,d))

# Verificaciones algebraicas reales
S.check_zero(
    "check_P_antisym_ab",
    P_up(a, b, c, d) + P_up(b, a, c, d),
    "P^{abcd}+P^{bacd}=0"
)
S.check_zero(
    "check_P_antisym_cd",
    P_up(a, b, c, d) + P_up(a, b, d, c),
    "P^{abcd}+P^{abdc}=0"
)
S.check_zero(
    "check_P_pair_exchange",
    P_up(a, b, c, d) - P_up(c, d, a, b),
    "P^{abcd}-P^{cdab}=0"
)


#### Coeficiente antes de imponer simetrías
Objeto reutilizable: `S['Q_naive']`

#### Después de antisimetrizar el primer par
Objeto reutilizable: `S['Q_antisym_ab']`

#### Después de antisimetrizar ambos pares
Objeto reutilizable: `S['Q_antisym_ab_cd']`

#### Después de simetrizar el intercambio de pares
Objeto reutilizable: `S['dR_dRiemann']`

#### P^{abcd} calculado desde L_input
Objeto reutilizable: `S['P_abcd']`

#### Verificación: P^{abcd}+P^{bacd}=0
Objeto: `S['check_P_antisym_ab']`

#### Verificación: P^{abcd}+P^{abdc}=0
Objeto: `S['check_P_antisym_cd']`

#### Verificación: P^{abcd}-P^{cdab}=0
Objeto: `S['check_P_pair_exchange']`


# Etapa 2. Calcular $P^{ab}=\left(\partial L/\partial g_{ab}\right)_{R_{ijkl}}$ desde el input

Aquí tampoco se usa $P^{ab}=-2\mathcal R^{ab}$. Se deriva directamente:

1. se calcula $\partial g^{ij}/\partial g_{ab}$ respetando la simetría métrica;
2. se deriva la contracción de Ricci;
3. se multiplica por $dL/dR$.


In [20]:

p, q = tensor_indices("p q", M)
i, j, k, l = tensor_indices("i j k l", M)

S.put(
    "R_scalar_tensor",
    g(i, k) * g(j, l) * Riem(-i, -j, -k, -l)
)

S.put(
    "dR_dg_cov_raw",
    dginv_dgcov(i, k, p, q) * g(j, l) * Riem(-i, -j, -k, -l)
    + g(i, k) * dginv_dgcov(j, l, p, q) * Riem(-i, -j, -k, -l),
    simplify=False
)

S.put("dR_dg_cov", S["dR_dg_cov_raw"])
S.put("P_metric_ab", f1 * S["dR_dg_cov"])

S.show("R_scalar_tensor", "R escrito como contracción tensorial real")
S.show("dR_dg_cov_raw", "Derivada métrica antes de canonizar")
S.show("dR_dg_cov", "Derivada métrica canonizada")
S.show("P_metric_ab", "P^{ab} calculado directamente desde L_input")

# Simetría de P^{ab}
S.check_zero(
    "check_P_metric_symmetry",
    S["P_metric_ab"] - S["P_metric_ab"].xreplace({p:q, q:p}),
    "P^{ab}-P^{ba}=0"
)

P_METRIC_TEMPLATE_INDICES = (p, q)
P_METRIC_TEMPLATE = S["P_metric_ab"]

def P_metric_up(a, b):
    return P_METRIC_TEMPLATE.xreplace({p:a, q:b})


#### R escrito como contracción tensorial real
Objeto reutilizable: `S['R_scalar_tensor']`

#### Derivada métrica antes de canonizar
Objeto reutilizable: `S['dR_dg_cov_raw']`

#### Derivada métrica canonizada
Objeto reutilizable: `S['dR_dg_cov']`

#### P^{ab} calculado directamente desde L_input
Objeto reutilizable: `S['P_metric_ab']`

#### Verificación: P^{ab}-P^{ba}=0
Objeto: `S['check_P_metric_symmetry']`


# Etapa 3. Dos cálculos de $\mathcal L_\xi L$, ahora **con el $L$ del usuario**

La primera ruta deriva el escalar $L(R)$.

La segunda ruta calcula por separado:

$
P^{ab}\mathcal L_\xi g_{ab},
\qquad
P^{ijkl}\mathcal L_\xi R_{ijkl},
$

pero usando los $P$ que ya fueron calculados desde el input.


In [21]:

# ------------------------------------------------
# 3A. Primera ruta: L es escalar
# ------------------------------------------------
m, i, j, k, l = tensor_indices("m i j k l", M)

S.put(
    "nabla_R_from_Riemann",
    curvature_projector(i, j, k, l) * DRiem(-m, -i, -j, -k, -l)
)

S.put(
    "nabla_L",
    f1 * S["nabla_R_from_Riemann"]
)

S.put(
    "Lie_L_route_1",
    xi(m) * S["nabla_L"]
)

S.show("nabla_R_from_Riemann", "∇_m R calculado desde la contracción de Riemann")
S.show("nabla_L", "∇_m L calculado por regla de la cadena")
S.show("Lie_L_route_1", "Primera ruta para la derivada de Lie")

# ------------------------------------------------
# 3B. Derivada de Lie de la métrica
# ------------------------------------------------
a, b, m = tensor_indices("a b m", M)

S.put(
    "Lie_metric_ab",
    g(-m, -b) * Dxi(-a, m)
    + g(-a, -m) * Dxi(-b, m)
)

S.put(
    "Lie_metric_contraction",
    P_metric_up(a, b) * S["Lie_metric_ab"]
)

S.show("Lie_metric_ab", "L_xi g_ab construido tensorialmente")
S.show("Lie_metric_contraction", "P^{ab} L_xi g_ab canonizado")

# ------------------------------------------------
# 3C. Derivada de Lie del Riemann: cinco términos reales
# ------------------------------------------------
i, j, k, l, m = tensor_indices("i j k l m", M)

S.put(
    "lie_curv_transport",
    P_up(i, j, k, l) * xi(m) * DRiem(-m, -i, -j, -k, -l)
)

curv_terms = [
    P_up(i, j, k, l) * Riem(-m, -j, -k, -l) * Dxi(-i, m),
    P_up(i, j, k, l) * Riem(-i, -m, -k, -l) * Dxi(-j, m),
    P_up(i, j, k, l) * Riem(-i, -j, -m, -l) * Dxi(-k, m),
    P_up(i, j, k, l) * Riem(-i, -j, -k, -m) * Dxi(-l, m),
]

for nterm, term in enumerate(curv_terms, 1):
    S.put(f"lie_curv_term_{nterm}_raw", term, simplify=False)
    S.put(f"lie_curv_term_{nterm}", term)
    S.show(f"lie_curv_term_{nterm}_raw", f"Término de curvatura {nterm}, antes de canonizar")
    S.show(f"lie_curv_term_{nterm}", f"Término de curvatura {nterm}, canonizado")

# Probar que los cuatro términos son idénticos, uno por uno
for nterm in range(2, 5):
    S.check_zero(
        f"check_lie_curv_term_{nterm}_equals_1",
        S[f"lie_curv_term_{nterm}"] - S["lie_curv_term_1"],
        f"T_{nterm}-T_1=0"
    )

S.put("lie_curv_four_sum", sum(curv_terms, sp.S.Zero))
S.put("Lie_Riemann_contraction", S["lie_curv_transport"] + S["lie_curv_four_sum"])

S.show("lie_curv_four_sum", "Suma realmente calculada de los cuatro términos")
S.show("Lie_Riemann_contraction", "P^{ijkl} L_xi R_{ijkl}")

# ------------------------------------------------
# 3D. Segunda ruta completa y comparación
# ------------------------------------------------
S.put(
    "Lie_L_route_2",
    S["Lie_metric_contraction"] + S["Lie_Riemann_contraction"]
)

S.show("Lie_L_route_2", "Segunda ruta completa")

S.check_zero(
    "check_two_Lie_routes",
    S["Lie_L_route_2"] - S["Lie_L_route_1"],
    "Ruta 2 - Ruta 1 = 0"
)


#### ∇_m R calculado desde la contracción de Riemann
Objeto reutilizable: `S['nabla_R_from_Riemann']`

#### ∇_m L calculado por regla de la cadena
Objeto reutilizable: `S['nabla_L']`

#### Primera ruta para la derivada de Lie
Objeto reutilizable: `S['Lie_L_route_1']`

#### L_xi g_ab construido tensorialmente
Objeto reutilizable: `S['Lie_metric_ab']`

#### P^{ab} L_xi g_ab canonizado
Objeto reutilizable: `S['Lie_metric_contraction']`

...

#### Verificación: T_4-T_1=0
Objeto: `S['check_lie_curv_term_4_equals_1']`

#### Suma realmente calculada de los cuatro términos
Objeto reutilizable: `S['lie_curv_four_sum']`

#### P^{ijkl} L_xi R_{ijkl}
Objeto reutilizable: `S['Lie_Riemann_contraction']`

#### Segunda ruta completa
Objeto reutilizable: `S['Lie_L_route_2']`

#### Verificación: Ruta 2 - Ruta 1 = 0
Objeto: `S['check_two_Lie_routes']`


# Etapa 4. Calcular $\mathcal R^{ab}$ y obtener la identidad principal

$\mathcal R^{ab}$ se construye por contracción del $P^{abcd}$ que salió del input. Después se compara con $P^{ab}$, también calculado directamente.


In [22]:

a, b, i, j, k = tensor_indices("a b i j k", M)

S.put(
    "Rcal_up_ab",
    P_up(a, i, j, k) * Riem(b, -i, -j, -k)
)

# Bajar los índices para uso posterior
p, q = tensor_indices("p q", M)
Rcal_up_pq = S["Rcal_up_ab"].xreplace({a:p, b:q})
S.put(
    "Rcal_down_ab",
    g(-a, -p) * g(-b, -q) * Rcal_up_pq
)

S.show("Rcal_up_ab", "Rcal^{ab} calculado por contracción")
S.show("Rcal_down_ab", "Rcal_ab calculado bajando índices")

S.check_zero(
    "check_main_identity",
    P_metric_up(a, b) + 2*S["Rcal_up_ab"],
    "P^{ab}+2 Rcal^{ab}=0"
)

S.check_zero(
    "check_Rcal_symmetry",
    S["Rcal_up_ab"] - S["Rcal_up_ab"].xreplace({a:b, b:a}),
    "Rcal^{ab}-Rcal^{ba}=0"
)


#### Rcal^{ab} calculado por contracción
Objeto reutilizable: `S['Rcal_up_ab']`

#### Rcal_ab calculado bajando índices
Objeto reutilizable: `S['Rcal_down_ab']`

#### Verificación: P^{ab}+2 Rcal^{ab}=0
Objeto: `S['check_main_identity']`

#### Verificación: Rcal^{ab}-Rcal^{ba}=0
Objeto: `S['check_Rcal_symmetry']`


# Etapa 5. Variar $\sqrt{-g}L$ con objetos simbólicos

Se cambia de $\delta g_{ab}$ a $\delta g^{ab}$ mediante una contracción real. Luego se construye la variación de la densidad.


In [23]:

a, b, p, q = tensor_indices("a b p q", M)

# Convertir la derivada respecto de g_ab a la derivada respecto de g^ab
S.put(
    "P_metric_contravariant_variable_ab",
    -g(-a, -p) * g(-b, -q) * P_metric_up(p, q)
)

S.put(
    "delta_L_metric",
    S["P_metric_contravariant_variable_ab"] * H(a, b)
)

a, b, c, d = tensor_indices("a b c d", M)
S.put(
    "delta_L_curvature_unsplit",
    P_up(a, b, c, d) * dRiem(-a, -b, -c, -d)
)

S.put(
    "delta_L_total_unsplit",
    S["delta_L_metric"] + S["delta_L_curvature_unsplit"]
)

a, b = tensor_indices("a b", M)
S.put(
    "delta_sqrt_minus_g",
    -sp.Rational(1,2) * sqrtg * g(-a, -b) * H(a, b)
)

S.put(
    "delta_density_unsplit",
    S["delta_sqrt_minus_g"] * L_input
    + sqrtg * S["delta_L_total_unsplit"]
)

for key, title in [
    ("P_metric_contravariant_variable_ab", "∂L/∂g^{ab} obtenido por cambio de variable"),
    ("delta_L_metric", "Parte métrica de δL"),
    ("delta_L_curvature_unsplit", "Parte de curvatura de δL"),
    ("delta_L_total_unsplit", "δL completo antes de Palatini"),
    ("delta_sqrt_minus_g", "δ√(-g) como objeto tensorial"),
    ("delta_density_unsplit", "δ(√(-g)L) antes de separar δR"),
]:
    S.show(key, title)

# Verificar que la pieza métrica es 2 Rcal_ab δg^ab
a, b = tensor_indices("a b", M)
S.check_zero(
    "check_metric_variation_equals_2Rcal",
    S["P_metric_contravariant_variable_ab"] - 2*S["Rcal_down_ab"],
    "∂L/∂g^{ab} - 2 Rcal_ab = 0"
)


#### ∂L/∂g^{ab} obtenido por cambio de variable
Objeto reutilizable: `S['P_metric_contravariant_variable_ab']`

#### Parte métrica de δL
Objeto reutilizable: `S['delta_L_metric']`

#### Parte de curvatura de δL
Objeto reutilizable: `S['delta_L_curvature_unsplit']`

#### δL completo antes de Palatini
Objeto reutilizable: `S['delta_L_total_unsplit']`

#### δ√(-g) como objeto tensorial
Objeto reutilizable: `S['delta_sqrt_minus_g']`

#### δ(√(-g)L) antes de separar δR
Objeto reutilizable: `S['delta_density_unsplit']`

#### Verificación: ∂L/∂g^{ab} - 2 Rcal_ab = 0
Objeto: `S['check_metric_variation_equals_2Rcal']`


# Etapa 6. Separar $\delta R_{abcd}$

Se sustituye

$R_{abcd}=g_{ae}R^e{}_{bcd}$

y se calcula explícitamente la pieza que contiene $\delta g_{ae}$, ya expresada en términos de $H^{ab}\equiv\delta g^{ab}$.


In [24]:

a, b, c, d, e = tensor_indices("a b c d e", M)

# Pieza debida a δg_ae, con δg_ae ya sustituida por H^mn
S.put(
    "delta_R_split_metric_piece",
    P_up(a, b, c, d)
    * h_from_H(a, e)
    * Riem(e, -b, -c, -d)
)

# Segunda pieza, todavía en términos de δR^e_bcd
dRm = TensorHead(r"\delta\mathrm{R}_{\mathrm{mix}}", [M]*4, TensorSymmetry.no_symmetry(4))

S.put(
    "delta_R_split_connection_piece",
    P_up(a, b, c, d) * g(-a, -e) * dRm(e, -b, -c, -d)
)

S.show("delta_R_split_metric_piece", "Primera pieza de P δR, realmente contraída")
S.show("delta_R_split_connection_piece", "Segunda pieza de P δR")

# La primera pieza debe ser -Rcal_ab H^ab
a, b = tensor_indices("a b", M)
S.check_zero(
    "check_split_metric_piece",
    S["delta_R_split_metric_piece"] + S["Rcal_down_ab"]*H(a,b),
    "P δg R + Rcal_ab H^ab = 0"
)


#### Primera pieza de P δR, realmente contraída
Objeto reutilizable: `S['delta_R_split_metric_piece']`

#### Segunda pieza de P δR
Objeto reutilizable: `S['delta_R_split_connection_piece']`

#### Verificación: P δg R + Rcal_ab H^ab = 0
Objeto: `S['check_split_metric_piece']`


# Etapa 7. Identidad de Palatini: combinar los dos términos

Se reemplaza $\delta R^e{}_{bcd}$ por dos derivadas de $\delta\Gamma$. El CAS prueba que, después de la antisimetría de $P$, ambos aportes son iguales.


In [25]:

a, b, c, d, e = tensor_indices("a b c d e", M)

palatini_1 = P_up(a,b,c,d) * g(-a,-e) * DGamma(e,-c,-d,-b)
palatini_2 = -P_up(a,b,c,d) * g(-a,-e) * DGamma(e,-d,-c,-b)

S.put("palatini_term_1_raw", palatini_1, simplify=False)
S.put("palatini_term_2_raw", palatini_2, simplify=False)
S.put("palatini_term_1", palatini_1)
S.put("palatini_term_2", palatini_2)
S.put("palatini_sum", palatini_1 + palatini_2)

for key in ("palatini_term_1_raw", "palatini_term_2_raw", "palatini_term_1", "palatini_term_2", "palatini_sum"):
    S.show(key)

S.check_zero(
    "check_palatini_two_terms_equal",
    S["palatini_term_2"] - S["palatini_term_1"],
    "segundo término - primer término = 0"
)

S.check_zero(
    "check_palatini_sum_is_twice",
    S["palatini_sum"] - 2*S["palatini_term_1"],
    "suma - 2×primer término = 0"
)


#### palatini_term_1_raw
Objeto reutilizable: `S['palatini_term_1_raw']`

#### palatini_term_2_raw
Objeto reutilizable: `S['palatini_term_2_raw']`

#### palatini_term_1
Objeto reutilizable: `S['palatini_term_1']`

#### palatini_term_2
Objeto reutilizable: `S['palatini_term_2']`

#### palatini_sum
Objeto reutilizable: `S['palatini_sum']`

#### Verificación: segundo término - primer término = 0
Objeto: `S['check_palatini_two_terms_equal']`

#### Verificación: suma - 2×primer término = 0
Objeto: `S['check_palatini_sum_is_twice']`


# Etapa 8. Sustituir $\delta\Gamma$ y reducir los tres términos

Ahora sí se construye

$\nabla_c\delta\Gamma^e{}_{db}$

a partir de $DDh_{cd\,bi}\equiv\nabla_c\nabla_d\delta g_{bi}$. Los tres términos se guardan y canonizan por separado.


In [26]:

def DGamma_from_h(e, c, d, b):
    i = tensor_indices("DG_i", M)
    return sp.Rational(1,2) * g(e, i) * (
        DDh(-c, -d, -b, -i)
        + DDh(-c, -b, -d, -i)
        - DDh(-c, -i, -d, -b)
    )

a, b, c, d, e = tensor_indices("a b c d e", M)
expanded = 2 * P_up(a,b,c,d) * g(-a,-e) * DGamma_from_h(e,c,d,b)

S.put("after_dGamma_full_raw", expanded, simplify=False)
S.put("after_dGamma_full", expanded)

# Construir los tres aportes individualmente
i = tensor_indices("i", M)
dg1 = P_up(i,b,c,d) * DDh(-c,-d,-b,-i)
dg2 = P_up(i,b,c,d) * DDh(-c,-b,-d,-i)
dg3 = -P_up(i,b,c,d) * DDh(-c,-i,-d,-b)

for nterm, term in enumerate((dg1,dg2,dg3),1):
    S.put(f"dGamma_piece_{nterm}_raw", term, simplify=False)
    S.put(f"dGamma_piece_{nterm}", term)
    S.show(f"dGamma_piece_{nterm}_raw", f"Pieza {nterm} antes de canonizar")
    S.show(f"dGamma_piece_{nterm}", f"Pieza {nterm} canonizada")

S.show("after_dGamma_full_raw")
S.show("after_dGamma_full")

# Pruebas exactas:
S.check_zero(
    "check_dGamma_piece_1_vanishes",
    S["dGamma_piece_1"],
    "primera pieza = 0 por antisimetría/simetría"
)

S.check_zero(
    "check_dGamma_piece_2_equals_3",
    S["dGamma_piece_2"] - S["dGamma_piece_3"],
    "pieza 2 - pieza 3 = 0"
)

# Objetivo obtenido sin escribirlo de antemano:
j = tensor_indices("j", M)
S.put(
    "palatini_metric_second_derivative",
    2*P_up(i,b,j,d)*DDh(-j,-b,-d,-i)
)

S.check_zero(
    "check_after_dGamma_reduction",
    S["after_dGamma_full"] - S["palatini_metric_second_derivative"],
    "expansión completa - combinación reducida = 0"
)

S.show("palatini_metric_second_derivative", "Resultado calculado tras sustituir δΓ")


#### Pieza 1 antes de canonizar
Objeto reutilizable: `S['dGamma_piece_1_raw']`

#### Pieza 1 canonizada
Objeto reutilizable: `S['dGamma_piece_1']`

#### Pieza 2 antes de canonizar
Objeto reutilizable: `S['dGamma_piece_2_raw']`

#### Pieza 2 canonizada
Objeto reutilizable: `S['dGamma_piece_2']`

#### Pieza 3 antes de canonizar
Objeto reutilizable: `S['dGamma_piece_3_raw']`

...

#### after_dGamma_full
Objeto reutilizable: `S['after_dGamma_full']`

#### Verificación: primera pieza = 0 por antisimetría/simetría
Objeto: `S['check_dGamma_piece_1_vanishes']`

#### Verificación: pieza 2 - pieza 3 = 0
Objeto: `S['check_dGamma_piece_2_equals_3']`

#### Verificación: expansión completa - combinación reducida = 0
Objeto: `S['check_after_dGamma_reduction']`

#### Resultado calculado tras sustituir δΓ
Objeto reutilizable: `S['palatini_metric_second_derivative']`


# Etapa 9. Primera integración por partes, verificada por regla del producto

No se escribe una igualdad decorativa. Se construye el vector de borde $B_1^j$, se calcula su divergencia por regla del producto y se verifica algebraicamente la identidad.


In [27]:

def DP_up(e, a, b, c, d):
    """∇_e P^{abcd}, calculado desde el P del input."""
    return scalar_covd(f1, e) * P_PROJECTOR_TEMPLATE.xreplace(
        dict(zip(P_TEMPLATE_INDICES, (a,b,c,d)))
    )

def DDP_up(e, f, a, b, c, d):
    """∇_e∇_f P^{abcd}, calculado desde el P del input."""
    return scalar_hessian(f1, e, f) * P_PROJECTOR_TEMPLATE.xreplace(
        dict(zip(P_TEMPLATE_INDICES, (a,b,c,d)))
    )

i, b, j, d = tensor_indices("i b j d", M)

S.put(
    "ibp_start",
    2*P_up(i,b,j,d)*DDh(-j,-b,-d,-i)
)

# B_1^j = 2 P^{ibjd} ∇_b h_di
S.put(
    "ibp1_boundary_vector",
    2*P_up(i,b,j,d)*Dh(-b,-d,-i)
)

# ∇_j B_1^j por regla del producto:
S.put(
    "ibp1_divergence",
    2*DP_up(j,i,b,j,d)*Dh(-b,-d,-i)
    + 2*P_up(i,b,j,d)*DDh(-j,-b,-d,-i)
)

S.put(
    "ibp1_residual_positive",
    2*DP_up(j,i,b,j,d)*Dh(-b,-d,-i)
)

for key in ("ibp_start", "ibp1_boundary_vector", "ibp1_divergence", "ibp1_residual_positive"):
    S.show(key)

S.check_zero(
    "check_ibp1",
    S["ibp_start"]
    - (S["ibp1_divergence"] - S["ibp1_residual_positive"]),
    "integrando inicial - (divergencia - residuo) = 0"
)


#### ibp_start
Objeto reutilizable: `S['ibp_start']`

#### ibp1_boundary_vector
Objeto reutilizable: `S['ibp1_boundary_vector']`

#### ibp1_divergence
Objeto reutilizable: `S['ibp1_divergence']`

#### ibp1_residual_positive
Objeto reutilizable: `S['ibp1_residual_positive']`

#### Verificación: integrando inicial - (divergencia - residuo) = 0
Objeto: `S['check_ibp1']`


# Etapa 10. Segunda integración por partes, nuevamente calculada

Se renombra el residuo de la primera IBP, se construye $B_2^j$, se calcula $\nabla_jB_2^j$ y se verifica la segunda identidad.


In [28]:

c, i, j, d = tensor_indices("c i j d", M)

S.put(
    "ibp1_residual_negative_renamed",
    -2*DP_up(c,i,j,c,d)*Dh(-j,-d,-i)
)

# Comprobar que es exactamente el negativo del residuo anterior
S.check_zero(
    "check_residual_renaming",
    S["ibp1_residual_negative_renamed"] + S["ibp1_residual_positive"],
    "residuo renombrado + residuo positivo anterior = 0"
)

# B_2^j = 2 h_di ∇_c P^{ijcd}
S.put(
    "ibp2_boundary_vector",
    2*h(-d,-i)*DP_up(c,i,j,c,d)
)

# ∇_j B_2^j por regla del producto
S.put(
    "ibp2_divergence",
    2*Dh(-j,-d,-i)*DP_up(c,i,j,c,d)
    + 2*h(-d,-i)*DDP_up(j,c,i,j,c,d)
)

S.put(
    "ibp2_bulk_hcov",
    2*h(-d,-i)*DDP_up(j,c,i,j,c,d)
)

for key in (
    "ibp1_residual_negative_renamed",
    "ibp2_boundary_vector",
    "ibp2_divergence",
    "ibp2_bulk_hcov",
):
    S.show(key)

S.check_zero(
    "check_ibp2",
    S["ibp1_residual_negative_renamed"]
    - (-S["ibp2_divergence"] + S["ibp2_bulk_hcov"]),
    "residuo - (-divergencia + nuevo bulk) = 0"
)


#### Verificación: residuo renombrado + residuo positivo anterior = 0
Objeto: `S['check_residual_renaming']`

#### ibp1_residual_negative_renamed
Objeto reutilizable: `S['ibp1_residual_negative_renamed']`

#### ibp2_boundary_vector
Objeto reutilizable: `S['ibp2_boundary_vector']`

#### ibp2_divergence
Objeto reutilizable: `S['ibp2_divergence']`

#### ibp2_bulk_hcov
Objeto reutilizable: `S['ibp2_bulk_hcov']`

#### Verificación: residuo - (-divergencia + nuevo bulk) = 0
Objeto: `S['check_ibp2']`


# Etapa 11. Término de borde completo y conversión a $\delta g^{ab}$

El término de borde es la combinación que salió de las dos reglas del producto. No se introduce desde una fórmula final.


In [29]:

S.put(
    "delta_v_vector",
    S["ibp1_boundary_vector"] - S["ibp2_boundary_vector"]
)

S.show("delta_v_vector", "δv^j especializado al L_input")

# Convertir el bulk final de h_ab a H^ab = δg^ab
d, i, j, c = tensor_indices("d i j c", M)

S.put(
    "ibp2_bulk_Hup",
    2*h_from_H(d,i)*DDP_up(j,c,i,j,c,d)
)

S.show("ibp2_bulk_Hup", "Bulk después de las dos IBP, en función de δg^{ab}")

# Calcular directamente -2 ∇^m ∇^n P_amnb desde P:
a, b, m, n, p, q, r, s, t, u = tensor_indices("a b m n p q r s t u", M)

S.put(
    "minus2_double_divergence_P_ab",
    -2
    * g(m,p) * g(n,q)
    * g(-a,-r) * g(-m,-s) * g(-n,-t) * g(-b,-u)
    * DDP_up(p,q,r,s,t,u)
)

S.show(
    "minus2_double_divergence_P_ab",
    "-2 ∇^m∇^n P_amnb calculado desde P"
)

S.check_zero(
    "check_ibp_bulk_equals_double_divergence",
    S["ibp2_bulk_Hup"]
    - S["minus2_double_divergence_P_ab"]*H(a,b),
    "bulk de IBP - (-2∇∇P)_ab H^ab = 0"
)


#### δv^j especializado al L_input
Objeto reutilizable: `S['delta_v_vector']`

#### Bulk después de las dos IBP, en función de δg^{ab}
Objeto reutilizable: `S['ibp2_bulk_Hup']`

#### -2 ∇^m∇^n P_amnb calculado desde P
Objeto reutilizable: `S['minus2_double_divergence_P_ab']`

#### Verificación: bulk de IBP - (-2∇∇P)_ab H^ab = 0
Objeto: `S['check_ibp_bulk_equals_double_divergence']`


# Etapa 12. Ensamblar $E_{ab}$ usando exclusivamente piezas ya calculadas

Se suman:

1. $\mathcal R_{ab}$, obtenido de $P\cdot R$;
2. la variación del volumen;
3. el bulk que salió de las dos integraciones por partes.


In [30]:

a, b = tensor_indices("a b", M)

S.put(
    "E_ab_raw",
    S["Rcal_down_ab"]
    - sp.Rational(1,2)*g(-a,-b)*L_input
    + S["minus2_double_divergence_P_ab"]
)

S.show("E_ab_raw", "Tensor de campo construido paso a paso")

# Contracción con la variación, que es la pieza bulk de δA
S.put(
    "delta_action_bulk_integrand",
    sqrtg * S["E_ab_raw"] * H(a,b)
)

S.show("delta_action_bulk_integrand", "Integrando bulk final de δA")

# Comprobación independiente de la estructura derivativa usando solo
# derivadas escalares de f_R; es una validación posterior, no la fuente del cálculo.
box_f1 = (
    f2 * DDR(m,-m)
    + f3 * DR(m)*DR(-m)
)
hess_f1_ab = (
    f2 * DDR(-a,-b)
    + f3 * DR(-a)*DR(-b)
)
S.put(
    "derivative_piece_chain_rule_check",
    g(-a,-b)*box_f1 - hess_f1_ab
)

S.check_zero(
    "check_double_divergence_chain_rule",
    S["minus2_double_divergence_P_ab"]
    - S["derivative_piece_chain_rule_check"],
    "(-2∇∇P)_ab - expansión por regla de la cadena = 0"
)


#### Tensor de campo construido paso a paso
Objeto reutilizable: `S['E_ab_raw']`

#### Integrando bulk final de δA
Objeto reutilizable: `S['delta_action_bulk_integrand']`

#### Verificación: (-2∇∇P)_ab - expansión por regla de la cadena = 0
Objeto: `S['check_double_divergence_chain_rule']`


# Etapa 13. Diagnóstico de orden desde $\nabla_aP^{abcd}$

El notebook deriva $P$ y luego contrae el índice derivativo. El diagnóstico usa el resultado calculado, no una ecuación introducida a mano.


In [31]:

a, b, c, d = tensor_indices("a b c d", M)

S.put(
    "divergence_P_bcd",
    DP_up(a,a,b,c,d)
)

S.show("divergence_P_bcd", "∇_a P^{abcd} calculado")

display(Markdown("### Diagnóstico automático"))
display(sp.Eq(sp.Symbol("f_RR"), f2))

if sp.simplify(f2) == 0:
    display(Markdown(
        "**Segundo orden:** el resultado calculado para `S['divergence_P_bcd']` "
        "se anula identitariamente."
    ))
    assert S["divergence_P_bcd"] == 0
else:
    display(Markdown(
        "**Genéricamente cuarto orden:** `S['divergence_P_bcd']` contiene "
        "gradientes de $R$, y `S['minus2_double_divergence_P_ab']` contiene "
        "el Hessiano de $R$."
    ))


#### ∇_a P^{abcd} calculado
Objeto reutilizable: `S['divergence_P_bcd']`

### Diagnóstico automático

**Segundo orden:** el resultado calculado para `S['divergence_P_bcd']` se anula identitariamente.


# Cómo reutilizar cualquier paso

No hay que volver a deducir nada ni copiar LaTeX. Los objetos están en `S`.

Ejemplos:

```python
S["P_abcd"]
S["lie_curv_term_3"]
S["after_dGamma_full"]
S["ibp1_divergence"]
S["delta_v_vector"]
S["minus2_double_divergence_P_ab"]
S["E_ab_raw"]
```

Puedes hacer nuevas operaciones:

```python
tsimplify(S["lie_curv_term_4"] - S["lie_curv_term_1"])
tsimplify(S["E_ab_raw"])
sp.diff(S["L_input"], R, 4)
```


In [32]:

display(Markdown(f"### Se almacenaron {len(S)} objetos simbólicos"))

# Mostrar las claves para que cualquier expresión pueda recuperarse.
for key in S.keys():
    print(key)


### Se almacenaron 91 objetos simbólicos

L_input
f_R
f_RR
f_RRR
Q_naive
Q_antisym_ab
Q_antisym_ab_cd
dR_dRiemann
P_abcd
check_P_antisym_ab
check_P_antisym_cd
check_P_pair_exchange
R_scalar_tensor
dR_dg_cov_raw
dR_dg_cov
P_metric_ab
check_P_metric_symmetry
nabla_R_from_Riemann
nabla_L
Lie_L_route_1
Lie_metric_ab
Lie_metric_contraction
lie_curv_transport
lie_curv_term_1_raw
lie_curv_term_1
lie_curv_term_2_raw
lie_curv_term_2
lie_curv_term_3_raw
lie_curv_term_3
lie_curv_term_4_raw
lie_curv_term_4
check_lie_curv_term_2_equals_1
check_lie_curv_term_3_equals_1
check_lie_curv_term_4_equals_1
lie_curv_four_sum
Lie_Riemann_contraction
Lie_L_route_2
check_two_Lie_routes
Rcal_up_ab
Rcal_down_ab
check_main_identity
check_Rcal_symmetry
P_metric_contravariant_variable_ab
delta_L_metric
delta_L_curvature_unsplit
delta_L_total_unsplit
delta_sqrt_minus_g
delta_density_unsplit
check_metric_variation_equals_2Rcal
delta_R_split_metric_piece
delta_R_split_connection_piece
check_split_metric_piece
palatini_term_1_raw
palatini_term_2_raw
palatini_t